<h1>Chapter 1 - Introduction to Language Models</h1>
<i>Exploring the exciting field of Language AI</i>


<a href="https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961"><img src="https://img.shields.io/badge/Buy%20the%20Book!-grey?logo=amazon"></a>
<a href="https://www.oreilly.com/library/view/hands-on-large-language/9781098150952/"><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="https://github.com/HandsOnLLM/Hands-On-Large-Language-Models"><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HandsOnLLM/Hands-On-Large-Language-Models/blob/main/chapter01/Chapter%201%20-%20Introduction%20to%20Language%20Models.ipynb)

---

This notebook is for Chapter 1 of the [Hands-On Large Language Models](https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961) book by [Jay Alammar](https://www.linkedin.com/in/jalammar) and [Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/).

---

<a href="https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961">
<img src="https://raw.githubusercontent.com/HandsOnLLM/Hands-On-Large-Language-Models/main/images/book_cover.png" width="350"/></a>


### [OPTIONAL] - Installing Packages on <img src="https://colab.google/static/images/icons/colab.png" width=100>

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** the following codeblock to install the dependencies for this chapter:

---

💡 **NOTE**: We will want to use a GPU to run the examples in this notebook. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**.

---

In [1]:
# %%capture
# !pip install transformers==4.41.2 accelerate==0.31.0

# Phi-3

The first step is to load our model onto the GPU for faster inference. Note that we load the model and tokenizer separately (although that isn't always necessary).

In [2]:
#######################################
## CUDA 계열은 아래 코드를 사용
#######################################

#from transformers import AutoModelForCausalLM, AutoTokenizer

# Load model and tokenizer
#model = AutoModelForCausalLM.from_pretrained(
    #"microsoft/Phi-3-mini-4k-instruct",
    #device_map="cuda",
    #torch_dtype="auto",
    #trust_remote_code=False,
#)
#tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

#######################################
## M 계열의 macbook은 아래 코드를 사용
#######################################

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# 1. 모델 식별자 설정: Microsoft에서 공개한 경량 LLM인 Phi-3 mini 모델 (매개변수 약 38억 개)
model_id = "microsoft/Phi-3-mini-4k-instruct"

# 2. 인과적 언어 모델(Causal LM) 로드 및 M4 Mac 최적화 설정
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    
    # [장치 할당] "mps"는 Metal Performance Shaders의 약자로, 
    # NVIDIA의 "cuda"와 달리 Apple Silicon(M1~M4)의 GPU 가속을 사용하게 합니다.
    device_map="mps", 
    
    # [데이터 타입] torch.float16(Half Precision)은 연산 속도를 높이고 메모리 점유율을 50% 줄여줍니다.
    # M4 칩의 통합 메모리 구조에서 가장 효율적인 성능을 발휘하는 규격입니다.
    torch_dtype=torch.float16, 
    
    # [커스텀 코드 허용] Phi-3와 같이 최신 아키텍처를 가진 모델들은 
    # 로컬 라이브러리에 없는 특수한 레이어 구조를 포함할 수 있으므로, 
    # 모델 저장소에 포함된 실행 코드를 신뢰하고 실행하도록 설정합니다.
    trust_remote_code=True,
)

# 3. 토크나이저 로드: 텍스트를 모델이 이해할 수 있는 숫자(ID) 시퀀스로 변환하는 도구
# 모델과 쌍을 이루는 전용 토크나이저를 다운로드하거나 캐시에서 불러옵니다.
tokenizer = AutoTokenizer.from_pretrained(model_id)

/Users/fharenheit/.pyenv/versions/3.11.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`flash-attention` package not found, consider installing for better performance: No module named 'flash_attn'.
Current `flash-attention` does not support `window_size`. Either upgrade or use `attn_implementation='eager'`.
Loading checkpoint shards: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:00<00:00,  3.07it/s]
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Although we can now use the model and tokenizer directly, it's much easier to wrap it in a `pipeline` object:

In [5]:
from transformers import pipeline

# 1. 텍스트 생성 파이프라인 구축
# pipeline은 모델 로드, 토큰화, 추론, 결과 후처리를 하나로 묶어주는 편리한 도구입니다.
generator = pipeline(
    "text-generation",      # 작업 유형: 텍스트 생성 (LLM의 기본 역할)
    model=model,            # 위에서 로드한 M4 GPU(MPS) 최적화 모델 사용
    tokenizer=tokenizer,    # 모델과 쌍을 이루는 전용 토크나이저 연결
    
    # [출력 제어] True일 경우 입력한 질문(Prompt)까지 결과에 포함합니다. 
    # False로 설정하여 모델이 새로 생성한 답변만 깔끔하게 받아봅니다.
    return_full_text=False, 
    
    # [길이 제한] 생성될 답변의 최대 토큰(단어 조각) 수를 500개로 제한합니다.
    # 너무 작으면 답변이 끊기고, 너무 크면 M4의 메모리를 많이 점유할 수 있습니다.
    max_new_tokens=500, 
    
    # [생성 전략] False(Greedy Search)로 설정하면 모델이 가장 확률이 높은 단어만 선택합니다.
    # 결과가 매우 일관되고 논리적이지만, 창의성이 다소 부족할 수 있습니다. 
    # (True로 설정 시 온도를 조절하여 더 창의적인 답변이 가능합니다.)
    do_sample=False 
)

Finally, we create our prompt as a user and give it to the model:

In [6]:
# 1. 메시지 구성 (사용자 질문 정의)
# 리스트와 딕셔너리 형태의 구조는 대화형 AI(Chat model)가 
# '누가(role)' '무슨 말(content)'을 했는지 맥락을 파악하게 돕습니다.
messages = [
    {
        "role": "user",          # 발화자 역할: 사용자(User)
        "content": "Create a funny joke about chickens." # 전달할 구체적인 질문 내용
    }
]

# 2. 결과 생성 (Inference 실행)
# 앞서 정의한 generator 파이프라인에 메시지를 전달합니다.
# 내부적으로는 모델이 이해할 수 있는 특수 토크(예: <|user|>, <|end|>)가 자동으로 추가됩니다.
output = generator(messages)

# 3. 결과 출력
# generator의 반환값은 리스트 내 딕셔너리 형태이므로, 
# 가장 첫 번째 결과([0])에서 생성된 텍스트(["generated_text"])만 추출하여 화면에 보여줍니다.
print(output[0]["generated_text"])

 Why did the chicken join the band? Because it had the drumsticks!
